# C1 — visible-prefix calibration: ceiling and transfer probe

**Append this after P6 in your v22 OOF fork.** It needs the patched `_field_query`
(self-exclusion) or every candidate reads its own answer off its own lateral and
the whole probe is meaningless.

## What prefix calibration is

For each well, hide a suffix of the **known** zone, re-run the model as if the
blind zone started earlier, and score candidates against the `TVT_input` truth you
just hid. Whichever candidate wins that self-audit is the one you trust on the
real blind zone. No labels needed — it works identically on test wells, which is
the entire point.

## Why this cell is not the selector

The method needs two things to be true, and only one of them is obvious:

1. **Headroom.** Different configurations of v22 must actually win on different
   wells. If one config dominates everywhere, selection has nothing to do.
2. **Transfer.** Candidate ranking on the *pseudo*-blind zone must predict ranking
   on the *real* blind zone. This is the shaky one: the pseudo zone sits right
   next to the anchor where drift hasn't accumulated, while the real blind zone
   runs to the toe. Being best near the heel doesn't guarantee being best at the
   toe.

C1 measures both before you invest in the machinery. If (1) is small or (2) is
near zero, prefix calibration cannot work here and you stop.

## The four candidates

Deliberately cheap — config levers only, no new code paths:

| candidate | change | when it should win |
|---|---|---|
| `default` | current CONFIG | baseline |
| `prior_free` | `w_b = 0.8` | structural prior misleads |
| `no_field` | field blend bypassed | field support is poor or wrong |
| `no_spatial` | `w_spatial = 0.0` | local dip estimate is bad |

`default` is flattered here — CONFIG was tuned on these wells — which makes the
ceiling a **conservative** estimate. Good direction to be wrong in.

## Cost

`2 × n_candidates` model runs per well (once on the pseudo problem, once on the
real one). At v22's ~5.4 s/well that's ~43 s/well, so the default `N_WELLS = 80`
takes roughly 55 minutes on CPU. Lower `N_WELLS` to 40 for a ~28-minute read.

In [ ]:
# ===== C1: candidate ceiling + prefix-transfer probe =====
import time, contextlib, pickle
import numpy as np
from scipy.stats import spearmanr

N_WELLS  = 80        # wells to probe; 40 ~= 28 min, 80 ~= 55 min
CUT_FRAC = 0.70      # keep the first 70% of the known zone, hide the rest
SEED     = 0

assert {'P1','P2','P3'} <= _PATCH_APPLIED, \
    'run P1-P3 first — without self-exclusion every candidate reads its own answer'

_MISSING = object()

CANDIDATES = {
    'default':    {},
    'prior_free': {'w_b': 0.8},
    'no_field':   {'__no_field': True},
    'no_spatial': {'w_spatial': 0.0},
}
CAND = list(CANDIDATES)


@contextlib.contextmanager
def _with_cfg(ov):
    """Temporarily override CONFIG keys and/or bypass the field blend."""
    global field_blend
    saved = {k: CONFIG.get(k, _MISSING) for k in ov if not k.startswith('__')}
    saved_fb = field_blend
    try:
        for k, v in ov.items():
            if not k.startswith('__'):
                CONFIG[k] = v
        if ov.get('__no_field'):
            field_blend = lambda h, pred, diag, arr=None: pred
        yield
    finally:
        for k, v in saved.items():
            if v is _MISSING:
                CONFIG.pop(k, None)
            else:
                CONFIG[k] = v
        field_blend = saved_fb


def _make_prefix_problem(h, cut_frac):
    """Hide the tail of the known zone. Returns (h2, hidden_mask) or (None, None)."""
    tin = h['TVT_input'].values.astype(float)
    known = np.where(np.isfinite(tin))[0]
    if len(known) < 200:
        return None, None
    kc = int(known[int(len(known) * cut_frac)])
    h2 = h.copy()
    v = tin.copy()
    hide = np.zeros(len(v), bool)
    hide[kc:] = True
    hide &= np.isfinite(tin)              # only hide originally-known rows
    if hide.sum() < 50:
        return None, None
    v[hide] = np.nan
    h2['TVT_input'] = v
    h2.attrs['well'] = h.attrs.get('well')   # keep self-exclusion alive
    return h2, hide


rng = np.random.default_rng(SEED)
_all = wells('train')
probe_wells = [_all[i] for i in sorted(rng.choice(len(_all),
                size=min(N_WELLS, len(_all)), replace=False))]

E_pre  = {c: {} for c in CAND}      # err on the hidden prefix slice
E_bli  = {c: {} for c in CAND}      # err on the real blind zone
rows, t0 = [], time.time()

for n, w in enumerate(probe_wells):
    try:
        h, t = load_well('train', w)
        h.attrs['well'] = w
        truth = h['TVT'].values.astype(float)
        tin   = h['TVT_input'].values.astype(float)
        blind = ~np.isfinite(tin)
        mb    = blind & np.isfinite(truth)
        if mb.sum() < 1:
            continue
        h2, hide = _make_prefix_problem(h, CUT_FRAC)
        if h2 is None:
            continue

        ok = True
        for c in CAND:
            with _with_cfg(CANDIDATES[c]):
                p_pre, _, _ = predict_well_diag(h2, t)      # pseudo problem
                p_bli, _, _ = predict_well_diag(h,  t)      # real problem
            p_pre = np.asarray(p_pre, float); p_bli = np.asarray(p_bli, float)
            g1 = hide & np.isfinite(p_pre) & np.isfinite(tin)
            g2 = mb   & np.isfinite(p_bli)
            if g1.sum() < 20 or g2.sum() < 1:
                ok = False; break
            E_pre[c][w] = float(np.abs(p_pre[g1] - tin[g1]).mean())
            E_bli[c][w] = float(np.abs(p_bli[g2] - truth[g2]).mean())
        if not ok:
            for c in CAND:
                E_pre[c].pop(w, None); E_bli[c].pop(w, None)
            continue
        rows.append(w)
    except Exception as e:
        for c in CAND:
            E_pre[c].pop(w, None); E_bli[c].pop(w, None)
        print('  skip %s: %r' % (w, e)[:110])

    if (n + 1) % 10 == 0:
        el = time.time() - t0
        print('%3d/%d wells  [%.0fs, ~%.0fs left]'
              % (n + 1, len(probe_wells), el,
                 el / (n + 1) * (len(probe_wells) - n - 1)), flush=True)
        pickle.dump({'pre': E_pre, 'bli': E_bli, 'wells': rows},
                    open('v22_prefix_probe.pkl', 'wb'))

pickle.dump({'pre': E_pre, 'bli': E_bli, 'wells': rows,
             'cut_frac': CUT_FRAC}, open('v22_prefix_probe.pkl', 'wb'))

# ---------------------------------------------------------------- analysis
W = rows
B = np.array([[E_bli[c][w] for c in CAND] for w in W])   # wells x candidates
P = np.array([[E_pre[c][w] for c in CAND] for w in W])
i_def = CAND.index('default')

print('\n' + '=' * 68)
print('%d wells, %d candidates, %.0f min' % (len(W), len(CAND), (time.time()-t0)/60))
print('\n%-14s %10s %10s %10s' % ('candidate', 'blind MAE', 'prefix MAE', 'win %'))
print('-' * 48)
best_i = B.argmin(1)
for j, c in enumerate(CAND):
    print('%-14s %10.3f %10.3f %9.1f%%'
          % (c, B[:, j].mean(), P[:, j].mean(), 100 * (best_i == j).mean()))

d_err  = B[:, i_def].mean()
oracle = B.min(1).mean()
sel_i  = P.argmin(1)                                  # pick by prefix
sel_err = B[np.arange(len(W)), sel_i].mean()
ceiling, captured = d_err - oracle, d_err - sel_err

rho = np.array([spearmanr(P[i], B[i]).statistic for i in range(len(W))])
rho = rho[np.isfinite(rho)]

print('\n%-34s %8.3f' % ('default', d_err))
print('%-34s %8.3f   (gain %.3f)' % ('ORACLE  (perfect selection)', oracle, ceiling))
print('%-34s %8.3f   (gain %.3f)' % ('prefix-SELECTED', sel_err, captured))
print('\nper-well spearman(prefix rank, blind rank): mean %.3f  median %.3f'
      % (rho.mean(), np.median(rho)))
print('  fraction of wells where prefix picked the true best: %.1f%%'
      % (100 * (sel_i == best_i).mean()))

print('\n' + '=' * 68)
print('VERDICT')
if ceiling < 0.20:
    print('  STOP. Even perfect per-well selection gains only %.3f — under the' % ceiling)
    print('  0.20 noise floor. These candidates are too similar to choose between.')
    print('  Either widen the candidate set or abandon prefix calibration.')
elif rho.mean() < 0.15:
    print('  STOP. Ceiling is %.3f but prefix rank barely predicts blind rank' % ceiling)
    print('  (spearman %.3f). The pseudo zone sits by the anchor, the real zone' % rho.mean())
    print('  runs to the toe — winning near the heel says nothing about the toe.')
    print('  Try a smaller CUT_FRAC (0.5) to push the pseudo zone further out.')
elif captured < 0.20:
    print('  MARGINAL. Ceiling %.3f is real but selection captured only %.3f.'
          % (ceiling, captured))
    print('  Worth one retry at CUT_FRAC=0.50 before giving up.')
else:
    print('  PROCEED. Ceiling %.3f, captured %.3f (%.0f%% of it) by honest'
          % (ceiling, captured, 100 * captured / max(ceiling, 1e-9)))
    print('  prefix selection. Build C2 and run it over the test wells.')
print('=' * 68)


## Reading it

**`win %`** tells you whether the candidates are meaningfully different. If
`default` wins 90%+ of wells, the others aren't real alternatives and you should
widen the set — a much stronger `w_b`, a different `emis_clip`, `em_iters=2` —
before concluding anything.

**The three numbers that matter** are `default`, `ORACLE`, and `prefix-SELECTED`.
Oracle is what a cheat gets and bounds everything. Prefix-selected is what you can
actually have. The gap between them is how much the pseudo-zone proxy costs you.

**The spearman is the real test of the method.** It asks directly whether being
good near the heel predicts being good at the toe. If it's near zero, no amount
of engineering fixes it, because the premise is false for this problem.

**If it says STOP on transfer, try `CUT_FRAC = 0.50` first.** Hiding half the
known zone pushes the pseudo-blind zone further from the anchor and makes it a
harder, more representative proxy. That single knob is the difference between a
proxy that tests what you care about and one that doesn't.

## What C2 would add

Only worth writing if C1 proceeds. It would apply the selection at inference on
test wells: for each well, run the candidates on its own hidden-prefix problem,
pick the winner, run that config for real, and write the submission. Roughly
`n_candidates + 1` runs per test well — about 20 minutes for 200 wells, still CPU
only, still no submission slot until you choose to use one.

One caution for later: selecting per well on a noisy proxy adds variance. With
four candidates and ~80 pseudo-blind stations per well, some of the apparent
selection gain will be the selector fitting noise. C2 should compare against a
shrunk variant — only switch away from `default` when the prefix margin exceeds a
threshold — rather than always taking the argmin.